> **Note:** The original `zameenData.csv` was not in Drive, so this solved copy uses a small synthetic Zameen-style sample with the same columns (created in the TODO 3 cell). All cleaning steps run on that sample exactly as they would on the real file.

# Data Cleansing Practice — Part 1
### Zameen.com Real Estate Data

This notebook mirrors a real data-cleaning workflow on scraped property-listing data (`zameenData.csv`) from Zameen.com, in preparation for a house price prediction model.

**How to use this notebook:** each **TODO** cell describes a task. Write your solution in the empty code cell right below it. Run cells in order — later tasks depend on earlier ones. Some tasks include hints in parentheses.


## Setup

**TODO 1:** Import the libraries you'll need: `numpy` as `np`, `pandas` as `pd`, and the built-in `re` module (for regular expressions).

In [1]:
import numpy as np
import pandas as pd
import re


**TODO 2:** Set pandas display options so that **all** rows and **all** columns are shown when you print a dataframe (`display.max_rows`, `display.max_columns`).

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


**TODO 3:** Load `zameenData.csv` into a dataframe called `df`, using `index_col=0`. The file has a typo in one column name — `pupose` instead of `purpose` — rename it. Display the first 5 rows.

In [3]:
# NOTE: zameenData.csv was not in Drive, so instead of pd.read_csv we build
# a small synthetic Zameen-style sample below with the same columns.
# The original column-name typo 'pupose' is kept on purpose, then renamed.
rng = np.random.default_rng(42)

# --- Base sample: 800 random listings (large enough for Part 2 groupings) ---
n_base = 800
neighbourhoods = ["DHA Phase 5", "Bahria Town", "Model Town", "Johar Town",
                  "Gulberg", "Cantt", "Valencia Town", "LDA Road"]
neighbourhood_p = [0.35, 0.20, 0.12, 0.10, 0.08, 0.05, 0.05, 0.05]
land_choices = ["House", "Flat", "Residential Plot", "Commercial Plot", "Building",
                "Shop", "Office", "Farm House", "Agricultural Land",
                "Industrial Land", "Factory", "Plot File", "Penthouse"]
land_p = [0.15, 0.12, 0.18, 0.06, 0.06, 0.06, 0.05, 0.05, 0.05, 0.04, 0.04, 0.05, 0.09]

# Random price strings in the three original units: Lakh, Crore, Arab
prices = []
for _ in range(n_base):
    r = rng.random()
    if r < 0.60:      # Crore
        prices.append(f"PKR{round(rng.uniform(0.4, 15), 1)} Crore")
    elif r < 0.92:    # Lakh
        prices.append(f"PKR{int(rng.integers(25, 96))} Lakh")
    else:             # Arab
        prices.append(f"PKR{round(rng.uniform(0.2, 1.0), 1)} Arab")

areas = rng.choice(["5 Marla", "10 Marla", "20 Marla", "1 Kanal", "2 Kanal",
                    "500 Sq. Yd.", "250 Sq. Yd.", "8 Marla"],
                   size=n_base, p=[0.28, 0.24, 0.14, 0.10, 0.05, 0.06, 0.06, 0.07])
df = pd.DataFrame({
    "agent": ["Agent " + ch for ch in rng.choice(list("ABCDEFGH"), size=n_base)],
    "name": [f"Listing {i}" for i in range(n_base)],
    "land_type": rng.choice(land_choices, size=n_base, p=land_p),
    "price": prices,
    "location": [n + ", Lahore, Punjab" for n in rng.choice(neighbourhoods, size=n_base, p=neighbourhood_p)],
    "baths": rng.choice(["1", "2", "3", "4", "5", "-"], size=n_base),
    "area": areas,
    "pupose": rng.choice(["For Sale", "For Sale", "For Sale", "For Rent"], size=n_base),
    "bed": rng.choice(["1", "2", "3", "4", "5", "-"], size=n_base),
    "added_days": rng.integers(1, 90, size=n_base),
})

# --- Hand-crafted rows that guarantee the tasks below have something to find ---
extra = [
    # 3 invalid 'purpose' values (a scraping error: addresses instead of For Rent/For Sale)
    {"agent": "Agent X", "name": "Bad purpose 1", "land_type": "House",
     "price": "PKR2.5 Crore", "location": "DHA Phase 5, Lahore, Punjab", "baths": "3",
     "area": "10 Marla", "pupose": "DHA Phase 5, Lahore, Punjab", "bed": "4", "added_days": 5},
    {"agent": "Agent X", "name": "Bad purpose 2", "land_type": "Flat",
     "price": "PKR90 Lakh", "location": "Johar Town, Lahore, Punjab", "baths": "2",
     "area": "10 Marla", "pupose": "Johar Town, Lahore", "bed": "3", "added_days": 12},
    {"agent": "Agent X", "name": "Bad purpose 3", "land_type": "House",
     "price": "PKR3.0 Crore", "location": "Bahria Town, Lahore, Punjab", "baths": "4",
     "area": "10 Marla", "pupose": "Bahria Town, Lahore", "bed": "5", "added_days": 8},
    # 3 Karachi rows (this dataset should only contain Lahore properties)
    {"agent": "Agent Y", "name": "Karachi 1", "land_type": "Flat",
     "price": "PKR80 Lakh", "location": "Clifton, Karachi, Sindh", "baths": "2",
     "area": "10 Marla", "pupose": "For Sale", "bed": "3", "added_days": 20},
    {"agent": "Agent Y", "name": "Karachi 2", "land_type": "House",
     "price": "PKR4.0 Crore", "location": "DHA, Karachi, Sindh", "baths": "4",
     "area": "1 Kanal", "pupose": "For Sale", "bed": "5", "added_days": 30},
    {"agent": "Agent Y", "name": "Karachi 3", "land_type": "Shop",
     "price": "PKR1.0 Crore", "location": "Gulshan, Karachi, Sindh", "baths": "1",
     "area": "5 Marla", "pupose": "For Rent", "bed": "0", "added_days": 15},
    # 3 comparable LDA Road / Residential Plot / 5 Marla listings (for the TODO 19 mean)
    {"agent": "Agent Z", "name": "LDA 1", "land_type": "Residential Plot",
     "price": "PKR1.5 Crore", "location": "LDA Road, Lahore, Punjab", "baths": "0",
     "area": "5 Marla", "pupose": "For Sale", "bed": "0", "added_days": 10},
    {"agent": "Agent Z", "name": "LDA 2", "land_type": "Residential Plot",
     "price": "PKR1.8 Crore", "location": "LDA Road, Lahore, Punjab", "baths": "0",
     "area": "5 Marla", "pupose": "For Sale", "bed": "0", "added_days": 11},
    {"agent": "Agent Z", "name": "LDA 3", "land_type": "Residential Plot",
     "price": "PKR1.2 Crore", "location": "LDA Road, Lahore, Punjab", "baths": "0",
     "area": "5 Marla", "pupose": "For Sale", "bed": "0", "added_days": 9},
    # 4 Building rows (for the TODO 39 Building price-per-marla)
    {"agent": "Agent W", "name": "Building 1", "land_type": "Building",
     "price": "PKR8.0 Crore", "location": "Gulberg, Lahore, Punjab", "baths": "6",
     "area": "10 Marla", "pupose": "For Sale", "bed": "6", "added_days": 40},
    {"agent": "Agent W", "name": "Building 2", "land_type": "Building",
     "price": "PKR12.0 Crore", "location": "Gulberg, Lahore, Punjab", "baths": "8",
     "area": "1 Kanal", "pupose": "For Sale", "bed": "8", "added_days": 50},
    {"agent": "Agent W", "name": "Building 3", "land_type": "Building",
     "price": "PKR5.0 Crore", "location": "Model Town, Lahore, Punjab", "baths": "4",
     "area": "10 Marla", "pupose": "For Sale", "bed": "4", "added_days": 25},
    {"agent": "Agent W", "name": "Building 4", "land_type": "Building",
     "price": "PKR6.5 Crore", "location": "Cantt, Lahore, Punjab", "baths": "5",
     "area": "10 Marla", "pupose": "For Sale", "bed": "5", "added_days": 35},
    # 2 extra Arab-priced rows (to double-check the Arab -> Crore conversion)
    {"agent": "Agent V", "name": "Arab 1", "land_type": "Commercial Plot",
     "price": "PKR1.2 Arab", "location": "DHA Phase 5, Lahore, Punjab", "baths": "0",
     "area": "1 Kanal", "pupose": "For Sale", "bed": "0", "added_days": 60},
    {"agent": "Agent V", "name": "Arab 2", "land_type": "House",
     "price": "PKR0.5 Arab", "location": "Bahria Town, Lahore, Punjab", "baths": "5",
     "area": "1 Kanal", "pupose": "For Sale", "bed": "5", "added_days": 55},
    # 1 exact duplicate of the first row (for the duplicates task)
    dict(df.iloc[0]),
    # 1 BAD row: unparsable price, index 18903 (as in the original task)
    {"agent": "Agent Z", "name": "Bad price", "land_type": "Residential Plot",
     "price": "PKR Call for Price", "location": "LDA Road, Lahore, Punjab", "baths": "0",
     "area": "5 Marla", "pupose": "For Sale", "bed": "0", "added_days": 7},
]
df = pd.concat([df, pd.DataFrame(extra)], ignore_index=True)
# Give the bad-price row the original index 18903 (all other rows keep 0..n-2)
df.index = list(range(len(df) - 1)) + [18903]

# Fix the typo in the column name: 'pupose' -> 'purpose'
df = df.rename(columns={"pupose": "purpose"})
print(df.head())


     agent       name          land_type        price  \
0  Agent B  Listing 0               Shop   PKR71 Lakh   
1  Agent E  Listing 1    Commercial Plot   PKR56 Lakh   
2  Agent G  Listing 2    Commercial Plot   PKR39 Lakh   
3  Agent A  Listing 3    Industrial Land  PKR0.8 Arab   
4  Agent A  Listing 4  Agricultural Land   PKR31 Lakh   

                        location baths      area   purpose bed  added_days  
0     Model Town, Lahore, Punjab     3  10 Marla  For Sale   4          19  
1    Bahria Town, Lahore, Punjab     1  10 Marla  For Sale   4          82  
2        Gulberg, Lahore, Punjab     2  10 Marla  For Sale   -          61  
3  Valencia Town, Lahore, Punjab     -   8 Marla  For Sale   4           6  
4  Valencia Town, Lahore, Punjab     1  10 Marla  For Sale   2          12  


**TODO 4:** Check the shape of `df`.

In [4]:
print(df.shape)


(817, 10)


## Duplicates

**TODO 5:** Drop duplicate rows from `df`.

In [5]:
# Drop exact duplicate rows
df = df.drop_duplicates()


**TODO 6:** Check the shape again to confirm duplicates were removed.

In [6]:
print(df.shape)


(816, 10)


## Dropping unneeded columns & fixing the `purpose` column

**TODO 7:** Look at the value counts of the `land_type` column, just to get a feel for the categories present.

In [7]:
print(df['land_type'].value_counts())


land_type
Residential Plot     141
House                141
Flat                  98
Penthouse             77
Building              52
Shop                  47
Office                47
Commercial Plot       44
Farm House            41
Agricultural Land     37
Industrial Land       35
Plot File             33
Factory               23
Name: count, dtype: int64


**TODO 8:** Drop the `agent` and `name` columns — they won't be useful for a price-prediction model.

In [8]:
# 'agent' and 'name' are not useful for price prediction
df = df.drop(columns=['agent', 'name'])


**TODO 9:** Look at the value counts of the `purpose` column. Do you see any values besides `"For Rent"` and `"For Sale"`?

In [9]:
print(df['purpose'].value_counts())


purpose
For Sale                       630
For Rent                       183
DHA Phase 5, Lahore, Punjab      1
Johar Town, Lahore               1
Bahria Town, Lahore              1
Name: count, dtype: int64


**TODO 10:** Some rows have invalid `purpose` values (a scraping error produced things like addresses instead of "For Rent"/"For Sale"). Select all rows where `purpose` is **neither** `"For Rent"` nor `"For Sale"`, and store them in a variable (e.g. `wrong_scrap`).

In [10]:
# Rows whose purpose is neither 'For Rent' nor 'For Sale' (scraping errors)
wrong_scrap = df[~df['purpose'].isin(['For Rent', 'For Sale'])]
print(wrong_scrap)


    land_type         price                     location baths      area  \
800     House  PKR2.5 Crore  DHA Phase 5, Lahore, Punjab     3  10 Marla   
801      Flat    PKR90 Lakh   Johar Town, Lahore, Punjab     2  10 Marla   
802     House  PKR3.0 Crore  Bahria Town, Lahore, Punjab     4  10 Marla   

                         purpose bed  added_days  
800  DHA Phase 5, Lahore, Punjab   4           5  
801           Johar Town, Lahore   3          12  
802          Bahria Town, Lahore   5           8  


**TODO 11:** Drop the rows found above from `df` (use their index).

In [11]:
df = df.drop(index=wrong_scrap.index)


**TODO 12:** Check the `purpose` value counts again — you should now only see `"For Rent"` and `"For Sale"`.

In [12]:
print(df['purpose'].value_counts())


purpose
For Sale    630
For Rent    183
Name: count, dtype: int64


**TODO 13:** Keep only rows where `purpose == "For Sale"` (our model is only for sale listings, not rentals). Check the resulting shape.

In [13]:
# Keep only sale listings
df = df[df['purpose'] == 'For Sale']
print(df.shape)


(630, 8)


## Price column

**TODO 14:** Look at the top 5 value counts of the raw `price` column. Notice the values look like `"PKR1.2 Crore"` and use **three different units**: Lakh, Crore, and Arab. There is also one bad/garbage value, at index `18903`.

In [14]:
print(df['price'].value_counts().head(5))


price
PKR0.3 Arab     10
PKR0.5 Arab      9
PKR9.0 Crore     8
PKR93 Lakh       8
PKR0.8 Arab      7
Name: count, dtype: int64


**TODO 15:** Write a function `convert_into_crore(text)` that:
- returns `None` if `text` is `None` or not a string,
- extracts the numeric amount and the unit (Lakh / Crore / Arab) from strings like `"PKR1.2 Crore"` using `re.search`,
- converts the amount into **Crore** units (1 Crore = 100 Lakh, 1 Arab = 100 Crore),
- returns the converted numeric value (or `None` if it can't parse the string).

(Hint: `re.search(r"PKR\d+\.?\d*\s*([a-zA-Z]+)", text)` gets you the unit; a similar pattern with `(\d+\.?\d*)` gets you the number.)

In [15]:
def convert_into_crore(text):
    # Return None for missing / non-string values
    if text is None or not isinstance(text, str):
        return None
    # Find the number and the unit, e.g. "PKR1.2 Crore"
    num_match = re.search(r"(\d+\.?\d*)", text)
    unit_match = re.search(r"PKR\d+\.?\d*\s*([a-zA-Z]+)", text)
    if num_match is None or unit_match is None:
        return None
    amount = float(num_match.group(1))
    unit = unit_match.group(1)
    if unit == "Lakh":
        return amount / 100  # 1 Crore = 100 Lakh
    if unit == "Crore":
        return amount
    if unit == "Arab":
        return amount * 100  # 1 Arab = 100 Crore
    return None


**TODO 16:** Apply `convert_into_crore` to the `price` column and insert the result as a new column called `price_cr` at column position 2 (use `df.insert`).

In [16]:
# Convert price to Crore and insert it as a new column at position 2
df.insert(2, 'price_cr', df['price'].apply(convert_into_crore))


**TODO 17:** Preview a random sample of 5 rows to sanity-check the new column.

In [17]:
print(df.sample(5, random_state=1))


    land_type          price  price_cr                       location baths  \
639      Shop     PKR43 Lakh      0.43    Bahria Town, Lahore, Punjab     1   
139     House   PKR4.0 Crore      4.00          Cantt, Lahore, Punjab     1   
88      House  PKR13.8 Crore     13.80  Valencia Town, Lahore, Punjab     5   
204  Building   PKR5.0 Crore      5.00    DHA Phase 5, Lahore, Punjab     2   
659      Flat  PKR11.5 Crore     11.50    Bahria Town, Lahore, Punjab     3   

            area   purpose bed  added_days  
639     20 Marla  For Sale   1          16  
139     10 Marla  For Sale   5          39  
88   250 Sq. Yd.  For Sale   5          18  
204  250 Sq. Yd.  For Sale   1          12  
659  250 Sq. Yd.  For Sale   5          66  


**TODO 18:** Filter and preview rows where `price` contains `"Arab"`, to double-check your Arab-to-Crore conversion looks right.

In [18]:
# Check the Arab -> Crore conversion
print(df[df['price'].str.contains('Arab', na=False)])


             land_type        price  price_cr                       location  \
3      Industrial Land  PKR0.8 Arab      80.0  Valencia Town, Lahore, Punjab   
75               House  PKR0.6 Arab      60.0          Cantt, Lahore, Punjab   
93              Office  PKR0.4 Arab      40.0    Bahria Town, Lahore, Punjab   
95           Penthouse  PKR0.3 Arab      30.0    DHA Phase 5, Lahore, Punjab   
97     Industrial Land  PKR0.9 Arab      90.0    DHA Phase 5, Lahore, Punjab   
120            Factory  PKR0.5 Arab      50.0    DHA Phase 5, Lahore, Punjab   
145          Penthouse  PKR0.3 Arab      30.0       LDA Road, Lahore, Punjab   
197    Commercial Plot  PKR0.3 Arab      30.0     Model Town, Lahore, Punjab   
206               Flat  PKR0.6 Arab      60.0     Johar Town, Lahore, Punjab   
212              House  PKR0.3 Arab      30.0    DHA Phase 5, Lahore, Punjab   
236         Farm House  PKR0.8 Arab      80.0          Cantt, Lahore, Punjab   
275          Plot File  PKR0.6 Arab     

**TODO 19:** The row at index `18903` has a bad `price_cr` value. Find comparable listings — same `location == "LDA Road, Lahore, Punjab"`, same `land_type == "Residential Plot"`, same `area == "5 Marla"` — and compute the **mean** `price_cr` for that group, rounded to 2 decimals.

In [19]:
# Comparable listings: same location, land type and area as the bad row
comparable = df[(df['location'] == 'LDA Road, Lahore, Punjab')
                & (df['land_type'] == 'Residential Plot')
                & (df['area'] == '5 Marla')]
print(comparable[['location', 'land_type', 'area', 'price', 'price_cr']])
mean_price = round(comparable['price_cr'].mean(), 2)
print('Mean price_cr for comparable listings:', mean_price)


                       location         land_type     area  \
806    LDA Road, Lahore, Punjab  Residential Plot  5 Marla   
807    LDA Road, Lahore, Punjab  Residential Plot  5 Marla   
808    LDA Road, Lahore, Punjab  Residential Plot  5 Marla   
18903  LDA Road, Lahore, Punjab  Residential Plot  5 Marla   

                    price  price_cr  
806          PKR1.5 Crore       1.5  
807          PKR1.8 Crore       1.8  
808          PKR1.2 Crore       1.2  
18903  PKR Call for Price       NaN  
Mean price_cr for comparable listings: 1.5


**TODO 20:** Use `.at[]` to overwrite the bad value at index `18903` in `price_cr` with the mean you just computed.

In [20]:
# Overwrite the bad value at index 18903 with the mean just computed
df.at[18903, 'price_cr'] = mean_price


## Location column

**TODO 21:** Write a function `get_loc(loc)` that strips the trailing `", Lahore, Punjab"` from a location string, returning just the neighborhood/area name.

In [21]:
def get_loc(loc):
    # Strip the trailing ", Lahore, Punjab" from a location string
    if isinstance(loc, str):
        return loc.replace(', Lahore, Punjab', '')
    return loc


**TODO 22:** Apply `get_loc` to the `location` column and insert the result as a new column called `locat`, at position 4.

In [22]:
# Insert the shortened location as a new column 'locat' at position 4
df.insert(4, 'locat', df['location'].apply(get_loc))


**TODO 23:** Preview a random sample row to check the result.

In [23]:
print(df.sample(1, random_state=1))


    land_type       price  price_cr                     location        locat  \
639      Shop  PKR43 Lakh      0.43  Bahria Town, Lahore, Punjab  Bahria Town   

    baths      area   purpose bed  added_days  
639     1  20 Marla  For Sale   1          16  


**TODO 24:** Check the dataframe shape and the `purpose` value counts again, just to confirm nothing broke.

In [24]:
print(df.shape)
print(df['purpose'].value_counts())


(630, 10)
purpose
For Sale    630
Name: count, dtype: int64


## Area column
`Marla`, `Kanal`, `Sq. Yd.`

**TODO 25:** Look at the top 5 value counts of the `area` column.

In [25]:
print(df['area'].value_counts().head(5))


area
5 Marla     181
10 Marla    158
20 Marla     83
1 Kanal      56
8 Marla      45
Name: count, dtype: int64


**TODO 26:** Write a function `convert_into_marla(text)` that converts an area string into a single unit, **Marla**:
- `1 Kanal = 20 Marla`
- `1 Sq. Yd. = (value * 9) / 225` Marla
- values already in Marla stay the same

(Hint: use regex to separate the numeric value from the unit text, e.g. `re.search(r"\d+\.?\d*", text)` for the number.)

In [26]:
def convert_into_marla(text):
    # Convert an area string into Marla (1 Kanal = 20 Marla, 1 Sq. Yd. = 9/225 Marla)
    if text is None or not isinstance(text, str):
        return None
    num_match = re.search(r"\d+\.?\d*", text)
    if num_match is None:
        return None
    value = float(num_match.group(0))
    if 'Kanal' in text:
        return value * 20
    if 'Sq. Yd.' in text:
        return (value * 9) / 225
    return value  # already in Marla


**TODO 27:** Apply `convert_into_marla` to `area` and insert the result as a new column `area_value`, at position 7.

In [27]:
# Convert area to Marla and insert it as a new column 'area_value' at position 7
df.insert(7, 'area_value', df['area'].apply(convert_into_marla))


**TODO 28:** Filter and preview rows where `area` contains `"Sq. Yd."`, to check your conversion looks correct.

In [28]:
# Check the Sq. Yd. -> Marla conversion
print(df[df['area'].str.contains('Sq. Yd.', na=False)])


             land_type          price  price_cr  \
17    Residential Plot     PKR38 Lakh      0.38   
48    Residential Plot  PKR10.9 Crore     10.90   
64          Farm House  PKR12.9 Crore     12.90   
66               House   PKR9.6 Crore      9.60   
88               House  PKR13.8 Crore     13.80   
89    Residential Plot   PKR0.9 Crore      0.90   
105              House   PKR2.2 Crore      2.20   
120            Factory    PKR0.5 Arab     50.00   
124             Office   PKR8.5 Crore      8.50   
126          Penthouse   PKR2.4 Crore      2.40   
129               Shop     PKR55 Lakh      0.55   
140              House  PKR11.3 Crore     11.30   
157    Commercial Plot   PKR8.0 Crore      8.00   
163   Residential Plot     PKR69 Lakh      0.69   
174  Agricultural Land  PKR14.3 Crore     14.30   
183   Residential Plot   PKR9.4 Crore      9.40   
190   Residential Plot   PKR9.0 Crore      9.00   
191              House   PKR9.4 Crore      9.40   
192   Residential Plot   PKR8.6

**TODO 29:** This dataset should only contain Lahore properties. Find rows where `location` contains `"Karachi"` and drop them from `df`.

In [29]:
# This dataset should only contain Lahore properties: drop Karachi rows
karachi_rows = df[df['location'].str.contains('Karachi', na=False)]
df = df.drop(index=karachi_rows.index)
print(df.shape)


(628, 11)


## Baths column

**TODO 30:** Check the value counts of `baths`.

In [30]:
print(df['baths'].value_counts())


baths
-    113
1    110
4    104
5    104
2    101
3     89
0      5
6      1
8      1
Name: count, dtype: int64


**TODO 31:** Replace `"-"` values in `baths` with `"0"`.

In [31]:
# Replace '-' in baths with '0'
df['baths'] = df['baths'].replace('-', '0')


**TODO 32:** Preview a random sample row.

In [32]:
print(df.sample(1, random_state=2))


    land_type          price  price_cr                    location  \
567  Building  PKR10.1 Crore      10.1  Johar Town, Lahore, Punjab   

          locat baths     area  area_value   purpose bed  added_days  
567  Johar Town     4  2 Kanal        40.0  For Sale   2          67  


## Bed column

**TODO 33:** Check the value counts of `bed`.

In [33]:
print(df['bed'].value_counts())


bed
4    111
1    108
2    105
5    105
3     98
-     94
0      5
6      1
8      1
Name: count, dtype: int64


**TODO 34:** Replace `"-"` values in `bed` with `"0"`.

In [34]:
# Replace '-' in bed with '0'
df['bed'] = df['bed'].replace('-', '0')


**TODO 35:** Preview the columns `land_type`, `area_value`, `price_cr`, and `baths` together, just to sanity-check everything so far.

In [35]:
print(df[['land_type', 'area_value', 'price_cr', 'baths']].sample(5, random_state=1))


            land_type  area_value  price_cr baths
639              Shop        20.0      0.43     1
139             House        10.0      4.00     1
88              House        10.0     13.80     5
204          Building        10.0      5.00     2
657  Residential Plot        20.0      0.75     4


## Land type column

**TODO 36:** Check the value counts of `land_type` again.

In [36]:
print(df['land_type'].value_counts())


land_type
House                108
Residential Plot     107
Flat                  73
Penthouse             64
Building              41
Commercial Plot       38
Office                38
Shop                  35
Farm House            30
Agricultural Land     27
Industrial Land       26
Plot File             22
Factory               19
Name: count, dtype: int64


**TODO 37:** Convert the `area_value` column to `float` dtype.

In [37]:
df['area_value'] = df['area_value'].astype(float)


**TODO 38:** Check `land_type` value counts once more.

In [38]:
print(df['land_type'].value_counts())


land_type
House                108
Residential Plot     107
Flat                  73
Penthouse             64
Building              41
Commercial Plot       38
Office                38
Shop                  35
Farm House            30
Agricultural Land     27
Industrial Land       26
Plot File             22
Factory               19
Name: count, dtype: int64


**TODO 39:** For rows where `land_type == "Building"`, compute the overall price-per-marla: sum of `price_cr` divided by sum of `area_value`.

In [39]:
# Overall price-per-marla for Building rows: total price / total area
building = df[df['land_type'] == 'Building']
price_per_marla_building = building['price_cr'].sum() / building['area_value'].sum()
print('Building price per marla (Crore):', round(price_per_marla_building, 4))


Building price per marla (Crore): 0.6353


**TODO 40:** Define a list `useful_land_types` containing: `'House'`, `'Flat'`, `'Building'`, `'Residential Plot'`, `'Commercial Plot'`, `'Plot File'`, `'Agricultural Land'`, `'Industrial Land'`, `'Factory'`, `'Farm House'`, `'Shop'`, `'Office'`. Keep only the rows whose `land_type` is in this list (this drops rare/tiny categories). Check the value counts of `land_type` afterward.

In [40]:
useful_land_types = ['House', 'Flat', 'Building', 'Residential Plot',
                     'Commercial Plot', 'Plot File', 'Agricultural Land',
                     'Industrial Land', 'Factory', 'Farm House', 'Shop', 'Office']
# Keep only the useful land types (drops rare/tiny categories)
df = df[df['land_type'].isin(useful_land_types)]
print(df['land_type'].value_counts())


land_type
House                108
Residential Plot     107
Flat                  73
Building              41
Commercial Plot       38
Office                38
Shop                  35
Farm House            30
Agricultural Land     27
Industrial Land       26
Plot File             22
Factory               19
Name: count, dtype: int64


## Save the cleaned data

**TODO 41:** Save the cleaned dataframe to `dataCleansing1_complete.csv`.

In [41]:
df.to_csv('dataCleansing1_complete.csv')
print('Saved cleaned data to dataCleansing1_complete.csv')


Saved cleaned data to dataCleansing1_complete.csv
